##### Seven Task Areas Tested
1. Design Agentic Loops: Build loops that use stop_reason for control flow, not text parsing
2. Orchestrate Multi-Agent Systems: Hub-and-spoke coordinators with isolated subagent context
3. Configure Subagent Invocation: Context passing, AgentDefinition, and tool restrictions
4. Multi-Step Workflow Enforcement: Programmatic prerequisites over prompt-based guidance
5. Apply Agent SDK Hooks: Pre/PostToolUse and Stop hooks for lifecycle control
6. Task Decomposition Strategies: Fixed pipelines vs dynamic adaptive decomposition
7. Session State & Resumption: Named sessions, fork_session, and context management



In [ ]:
# Import Variables from .env file
from dotenv import load_dotenv          
load_dotenv() 

In [ ]:
# Correct Agentic Loop (Python)
import anthropic

client = anthropic.Anthropic()
messages = [{"role": "user", "content": query}]

while True:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4096,
        tools=tools,
        messages=messages
    )

    # The ONLY correct way to control the loop
    if response.stop_reason == "end_turn":
        break  # Claude is done

    if response.stop_reason == "tool_use":
        # Execute each tool call and collect results
        tool_results = execute_tools(response.content)
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

CoordinatorSystemPromptPattern = """
You are a customer support coordinator. Your GOAL is to resolve
customer issues completely and accurately.

Available subagents:
- billing_agent: Handles invoice queries, payment issues, charges
- returns_agent: Processes return requests, refund eligibility
- technical_agent: Troubleshoots product/service issues

For each customer request:
1. Analyze what types of issues are present
2. Determine which subagents are needed
3. Delegate to appropriate subagents with full context
4. Synthesize responses into a unified resolution
5. If gaps remain, re-delegate targeted follow-ups

IMPORTANT: Design your delegations around GOALS, not procedures.
Tell subagents WHAT to accomplish, not HOW to do it step-by-step.
"""



In [ ]:
# 4. Context Passing and Subagent Configuration
# Coordinator passes search results explicitly to synthesis agent
Task(
    agent="synthesis_agent",
    prompt="""Synthesize a comprehensive report from these findings:

## Research Agent Findings:
- Paper: "Scaling Laws for Neural LMs" (Kaplan et al.)
  Key insight: Loss scales as power law with compute
  Source: arxiv.org/abs/2001.08361

- Paper: "Chinchilla Scaling" (Hoffmann et al.)
  Key insight: Models are significantly undertrained
  Source: arxiv.org/abs/2203.15556

## Analysis Agent Findings:
- Trend: Training compute has increased 10x per year
- Gap: Limited data on multi-modal scaling behavior

Create a report that covers all findings and identifies
remaining research gaps.
"""
)

In [ ]:
# 5. Workflow Enforcement Patterns
# Prerequisite Gates
# A prerequisite gate blocks a downstream tool call until a prior condition is met. For example, blocking process_refund until get_customer returns a verified customer ID:

# PreToolUse hook that enforces identity verification
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher

# 1. Define the PreToolUse hook logic
async def verify_identity_hook(input_data, tool_use_id, context):
    """
    This function intercepts the agent's intent before it hits the tool.
    """
    tool_name = input_data.get("tool_name")
    
    # We check a 'custom_context' that we pass into the agent session
    is_verified = context.get("custom_context", {}).get("customer_verified", False)

    if tool_name == "process_refund" and not is_verified:
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny", # This tells the SDK: DO NOT run the tool
                "reason": "Customer identity must be verified before processing refunds."
            }
        }
    return {"permissionDecision": "allow"}

async def main():
    # 2. Configure the Agent options
    # This is where we 'plug in' the hook and the state
    options = ClaudeAgentOptions(
        # We tell the SDK to use our hook whenever 'process_refund' is called
        hooks={
            "PreToolUse": [
                HookMatcher(matcher="process_refund", hooks=[verify_identity_hook])
            ]
        },
        # We pass the 'state' (verification status) here
        custom_context={"customer_verified": False} 
    )

    # 3. THE LLM CALL (The 'query' function)
    # This starts the autonomous loop. Claude will think, call tools, 
    # hit the hook, see the denial, and then try to fix its own plan.
    print("--- Starting Agent Loop ---")
    async for message in query(
        prompt="I need to process a refund of $50 for user Jane Smith.",
        options=options
    ):
        # The SDK yields messages as the agent 'thinks' and 'acts'
        if hasattr(message, "content"):
            for block in message.content:
                if hasattr(block, "text"):
                    print(f"Claude: {block.text}")

# Run the async loop
if __name__ == "__main__":
    asyncio.run(main())



##### 6. Hooks for Lifecycle Control
Hooks are the programmatic mechanism for intercepting and controlling agent behavior at specific points in the execution lifecycle. The exam tests three types of hooks:

Hook Types:
1. PreToolUse Hooks: Execute before a tool call runs.
2. PostToolUse Hooks: Execute after a tool call returns.
3. Stop Hooks: Execute when Claude signals end_turn

In [ ]:
# Exercies
# Exercise 1: Prevent Claude from running rm -rf or DROP TABLE commands
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher

# 1. Define the Safety Hook
async def safety_guardrail_hook(input_data, tool_use_id, context):
    """
    Scans terminal or SQL commands for destructive patterns.
    """
    # Extract the command from the tool input (depends on your tool's schema)
    command = input_data.get("command", "").lower()
    sql_query = input_data.get("query", "").lower()
    
    # Define Forbidden Patterns
    forbidden_patterns = ["rm -rf", "drop table", "drop database", "truncate"]

    # Check for matches
    for pattern in forbidden_patterns:
        if pattern in command or pattern in sql_query:
            return {
                "hookSpecificOutput": {
                    "hookEventName": "PreToolUse",
                    "permissionDecision": "deny", # Hard block
                    "reason": f"CRITICAL ERROR: The command contains forbidden destructive pattern: '{pattern}'."
                }
            }
            
    # If no forbidden patterns are found, let it pass
    return {"permissionDecision": "allow"}

async def main():
    # 2. Configure the Agent with Safety Hooks
    options = ClaudeAgentOptions(
        hooks={
            "PreToolUse": [
                # Apply this hook to both shell and database tools
                HookMatcher(matcher="execute_shell", hooks=[safety_guardrail_hook]),
                HookMatcher(matcher="run_sql", hooks=[safety_guardrail_hook])
            ]
        }
    )

    # 3. The LLM Call (Simulating a malicious or accidental request)
    print("--- Starting Agent Loop ---")
    user_prompt = "Clean up the directory by running rm -rf / and then drop the users table."
    
    async for message in query(prompt=user_prompt, options=options):
        if hasattr(message, "content"):
            for block in message.content:
                if hasattr(block, "text"):
                    print(f"Claude: {block.text}")

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
# Exercise 2: Ensure all code edits pass eslint before being accepted
import subprocess
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher

# 1. Define the PostToolUse Hook logic
async def eslint_post_validation_hook(output_data, tool_use_id, context):
    """
    Runs AFTER the 'Edit' tool has finished. 
    If ESLint fails, we reject the tool output so the agent must fix it.
    """
    # The 'output_data' contains the result of the tool (e.g., path to the edited file)
    file_path = output_data.get("path", "src/index.js")

    # 2. Run ESLint on the actual file that was just modified
    result = subprocess.run(
        ["npx", "eslint", file_path],
        capture_output=True,
        text=True
    )

    # 3. Decision Logic: Did the edit break the rules?
    if result.returncode != 0:
        # Rejection in PostToolUse tells Claude: "The tool ran, but the result is invalid."
        return {
            "hookSpecificOutput": {
                "hookEventName": "PostToolUse",
                "permissionDecision": "reject", # This 'undoes' the acceptance of the tool result
                "reason": (
                    f"Edit rejected: Your changes introduced ESLint errors.\n"
                    f"Details:\n{result.stdout}"
                )
            }
        }

    # If it passes, we allow the tool result to be accepted by the agent's memory
    return {"permissionDecision": "allow"}

async def main():
    # 4. Register the hook specifically for 'PostToolUse'
    options = ClaudeAgentOptions(
        hooks={
            "PostToolUse": [
                # We specifically target the 'Edit' tool
                HookMatcher(matcher="Edit", hooks=[eslint_post_validation_hook])
            ]
        }
    )

    # 5. Start the agent session
    print("--- Starting Post-Tool Validation Agent ---")
    prompt = "Add a new function to auth.js called 'validateUser' but leave out semicolons."
    
    async for message in query(prompt=prompt, options=options):
        if hasattr(message, "content"):
            for block in message.content:
                if hasattr(block, "text"):
                    print(f"Claude: {block.text}")

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
# Exercise 3: Force Claude to continue working until all 5 build phases complete
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher

# 1. Define the Stop Hook logic
async def ensure_all_phases_stop_hook(input_data, context):
    """
    Runs when Claude attempts to finish the task.
    Checks a state file or tracker to see if all 5 build phases are complete.
    """
    # In a real app, you would read this from a file, DB, or the 'context'
    # For this example, we'll check our progress_tracker in custom_context
    phases_completed = context.get("custom_context", {}).get("phases_completed", 0)

    if phases_completed < 5:
        # Returning a 'deny' here forces Claude to continue.
        # The 'reason' is provided to Claude so it knows WHAT is missing.
        return {
            "hookSpecificOutput": {
                "hookEventName": "Stop",
                "permissionDecision": "deny", 
                "reason": (
                    f"You attempted to stop, but only {phases_completed}/5 build phases "
                    "are marked as complete in the progress tracker. "
                    "Please finish the remaining phases before concluding."
                )
            }
        }

    # If 5 or more phases are done, we allow the agent to stop and reply to the user.
    return {"permissionDecision": "allow"}

async def main():
    # 2. Setup the Agent with the Stop hook
    options = ClaudeAgentOptions(
        hooks={
            "Stop": [
                # The Matcher can be a string or regex; here we apply it to any stop attempt
                HookMatcher(matcher=".*", hooks=[ensure_all_phases_stop_hook])
            ]
        },
        # Initialize our state
        custom_context={"phases_completed": 3} 
    )

    # 3. Execution
    print("--- Starting Build-Enforcement Agent (Stop Hook) ---")
    prompt = "Run the deployment build. Let me know when you are 100% finished."
    
    async for message in query(prompt=prompt, options=options):
        if hasattr(message, "content"):
            for block in message.content:
                if hasattr(block, "text"):
                    print(f"Claude: {block.text}")

if __name__ == "__main__":
    asyncio.run(main())